Here, you develop the 

In [2]:
import os.path
from scipy import *
import numpy as np
from numpy import *
from numpy import linalg as LA
from scipy import linalg as LA2
import sympy as sympy
import sys as sys
import time
import matplotlib.pyplot as plt
import itertools as it
from IPython.core.display import HTML


sys.path.append('/Users/sashacurcic/SashasDirectory/ANAG/FV_MG/')
from Modules import BasicTools as BT
from Modules import WaveTools as WT
from Modules import PlotTools as PT
from Modules import FFTTools as FFTT
from Modules import OperatorTools as OT
from Modules import GridTransferTools as GTT
from Modules import TestTools as TT
from Modules import SolverTools as ST
from Modules import WaveformTools as WFT

display(HTML("<style>pre { white-space: pre !important; }</style>"))
np.set_printoptions( linewidth = 10000, threshold = 100000)

In [3]:
ceqSwitch = 'c1neqc2'
contSwitch = 'cont'
IRT = 'IRT'
derivative = False
BooleAve = False
WaveFunc = WFT.Gauss
appendage = '-' + ceqSwitch + '-' + contSwitch + '-' + IRT

if (ceqSwitch == 'c1eqc2'):
    val2 = 1
else:
    val2 = 0.25

if (contSwitch == 'cont'):
    plotCont = True
else:
    plotCont = False

In [4]:
save = True
nh_min = 128
refRatio = 2
CFL = np.sqrt(0.5)
x_0 = 0.
x_1 = 0.5
Hans = False

deriv = 'UD'
RK = 4
order = 5

L = 1.
locs = [x_1]
epsilons = [val2, 1]# 11.68]
mus = [1, 1]#0.99837]

if (locs == []):
    field = 'E'
    AnalFunc = WFT.Advect
    PropFunc = ST.AdvectRHS
else:
    field = 'EB'
    AnalFunc = WFT.WaveEq
    PropFunc = ST.WaveEqRHS



Inititialize `Grid` objects on coarse, fine, and AMR grid.

In [6]:
omegaAMR = BT.Grid(nh_min)
refCells = nh_min // 2
finepart = list(np.arange(refCells) + int(nh_min // 4))
omegaAMR.AddPatch(refRatio, finepart)
degFreed = omegaAMR.degFreed
nh_max = omegaAMR.nh_max

omegaF = BT.Grid(nh_max)
omegaC = BT.Grid(nh_min)

Inititialize `PhysProps` objects on coarse, fine, and AMR grid.

In [8]:
physicsAMR = BT.PhysProps(omegaAMR, epsilons, mus, locs, L)
cVecAMR = physicsAMR.cVec
cMatAMR = physicsAMR.cMat
csAMR = physicsAMR.cs

physicsC = BT.PhysProps(omegaC, epsilons, mus, locs, L)
cVecC = physicsC.cVec
cMatC = physicsC.cMat
csC = physicsC.cs

physicsF = BT.PhysProps(omegaF, epsilons, mus, locs, L)
cVecF = physicsF.cVec
cMatF = physicsF.cMat
csF = physicsF.cs

c1 = csAMR[0]
mu1 = mus[0]
epsilon1 = epsilons[0]
if (locs ==[]):
    c2 = c1
    mu2 = mu1
    epsilon2 = epsilon1
else:
    c2 = csAMR[-1]
    mu2 = mus[-1]
    epsilon2 = epsilons[-1]

Harvesting values and creating operators.

In [10]:
# c = ConvertParams(permitivity, permeability, L)
# t, nt = ST.CalcTime(omegaC, CFL, csAMR[0], nt = nt) # CHANGE BACK TO OMEGAAMR!!!
sigma, mu = WFT.GaussParams(x_0, x_1 / 2)
wavesAMR = WT.MakeWaves(omegaAMR)
nullspace = OT.FindNullspace(omegaAMR, wavesAMR, Hans = Hans)
restrictOp = GTT.CoarsenOp(omegaAMR)

wavesF = WT.MakeWaves(omegaF)
wavesC = WT.MakeWaves(omegaC)

if (locs != []):
    wavesC = OT.Block(wavesC, var = 2)
    wavesF = OT.Block(wavesF, var = 2)
    wavesAMR = OT.Block(wavesAMR, var = 2)
    nullspace = OT.Block(nullspace, var = 2)
    restrictOp = OT.Block(restrictOp, var = 2)


# derivMat = TT.ExactSpatOp(omegaAMR)
# spatOp = -cMatF @ derivMat
# timePropOp = LA2.expm(t * spatOp)

# derivMatC = TT.ExactSpatOp(omegaC)
# spatOpC = -cMatC @ derivMatC
# timePropOpC = LA2.expm(t * spatOpC)

# opC = -cMatC @ OT.SpaceDeriv(omegaC, order, diff)
# opF = -cMatF @ OT.SpaceDeriv(omegaF, order, diff)
# opAMR = -cMatAMR @ OT.SpaceDeriv(omegaAMR, order, diff)

hC = np.concatenate((omegaC.h, omegaC.h))
hF = np.concatenate((omegaF.h, omegaF.h))
hAMR = np.concatenate((omegaAMR.h, omegaAMR.h))

degHalfC = int(nh_min / 2)
degHalfAMR = int(degFreed / 2)
degHalfF = int(nh_max / 2)

epsVecC = np.ones(nh_min, float)
epsVecAMR = np.ones(degFreed, float)
epsVecF = np.ones(nh_max, float)

epsVecC[:degHalfC] = epsilon1
epsVecC[degHalfC:] = epsilon2
epsVecF[:degHalfF] = epsilon1
epsVecF[degHalfF:] = epsilon2
epsVecAMR[:degHalfAMR] = epsilon1
epsVecAMR[degHalfAMR:] = epsilon2

muVecC = np.ones(nh_min, float)
muVecAMR = np.ones(degFreed, float)
muVecF = np.ones(nh_max, float)

muVecC[:degHalfC] = 1. / mu1
muVecC[degHalfC:] = 1. / mu2
muVecF[:degHalfF] = 1. / mu1
muVecF[degHalfF:] = 1. / mu2
muVecAMR[:degHalfAMR] = 1. / mu1
muVecAMR[degHalfAMR:] = 1. / mu2


muepsC = np.concatenate((epsVecC, muVecC))
muepsF = np.concatenate((epsVecF, muVecF))
muepsAMR = np.concatenate((epsVecAMR, muVecAMR))



# print(muepsC)
# print('')

Initialize waveform.

In [12]:
# # For Gaussian:
# waveInitC = WFT.Gauss(omegaC, physicsC, sigma, mu, BooleAve = False, deriv = False, cellAve = True)
# waveInitF = WFT.Gauss(omegaF, physicsF, sigma, mu)
args = [sigma, mu] # , 40]
waveInitC = WFT.InitCond(omegaC, physicsC, WaveFunc, args, deriv = derivative, BooleAve = BooleAve, field = field)
waveInitF = WFT.InitCond(omegaF, physicsF, WaveFunc, args, deriv = derivative, BooleAve = BooleAve, field = field)
waveInitAMR = WFT.InitCond(omegaAMR, physicsAMR, WaveFunc, args, deriv = derivative, BooleAve = BooleAve, field = field)
print(waveInitC)
print(len(waveInitC))
print('')
# waveInitF = np.append(waveInitF, waveInitF)
FCoefsC = FFTT.FourierCoefs(wavesC, waveInitC)
FCoefsF = FFTT.FourierCoefs(wavesF, waveInitF)
FCoefsC1 = np.asarray(np.append(FCoefsC, FCoefsC))


YOU ARE RUNNING GAUSS! False


YOU ARE RUNNING GAUSS! False


YOU ARE RUNNING GAUSS! False

[1.26435239e-13 5.18717204e-12 1.65713344e-10 4.12869559e-09 8.02376578e-08 1.21658742e-06 1.43945561e-05 1.32932952e-04 9.58371531e-04 5.39493064e-03 2.37173981e-02 8.14420884e-02 2.18471186e-01 4.57883740e-01 7.49847100e-01 9.59565043e-01 9.59565043e-01 7.49847100e-01 4.57883740e-01 2.18471186e-01 8.14420884e-02 2.37173981e-02 5.39493064e-03 9.58371531e-04 1.32932952e-04 1.43945561e-05 1.21658742e-06 8.02376578e-08 4.12869559e-09 1.65713344e-10 5.18717204e-12 1.26435239e-13 2.49543235e-15 0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00 0.00000000e+00 0.0000

If\
$u (x, 0) = \sum_{k = 0} a_{k} e^{2 \pi i k x}$,\
then the propagated incoming part of the wave is\
$u_{I} (x, t) = \sum_{k = 0} a_{k} e^{2 \pi i k (x - c_{1} t)} \Theta (x_{s} - x)$,\
and the reflected part of the wave is\
$u_{R} (x, t) = \frac{c_{2} - c_{1}}{c_{1} + c_{2}} \sum_{k = 0} a_{k} e^{2 \pi i k (2 x_{s} - x - c_{1} t)} \Theta (x_{s} - x)$.\
Thus, we can just multiply the original coefficients by an operator which contains the waves $e^{2 \pi i k (x - c_{1} t)}$ in the region of the first medium for the propagated wave and the waves $e^{2 \pi i k (2 x_{s} - x - c_{1} t)}$ in the region of the first medium for the reflected wave. This operator is zero in the region of the second medium.\
\
For the transmitted portion, I used $x'$ derived from the Method of Characteristics,\
$x'(x, t) = (x - c_{1} t) \Theta (x_{s} - x) + \frac{c_{1} (x - c_{2} t) + (c_{2} - c_{1}) x_{s}}{c_{2}} \Theta (x - x_{s}) \Theta (x_{s} + c_{2} t - x) + (x - c_{2} t) \Theta [x - (x_{s} + c_{2} t)]$,\
and plugged it into the initial wave like so:\
$u_{T} (x, t) = \frac{2 c_{1}}{c_{1} + c_{2}} u [x' (x, t), 0] \Theta (x - x_{s})$.\
The scaling factor out front is semi-derived, semi-guess. The value I derived that it should have been was $\frac{2 c_{2}}{c_{1} + c_{2}}$, but changing the numerator to $c_{1}$ made the wave look more continuous for some reason.


In [14]:
cs = physicsF.cs
t_snaps = 8
t_f = 3 / (4 * c1) # (3 / 8) * ((1 / c1) + (1 / c2))
t = 0
i = 0

ntVec = []#np.arange(iters) + 1
EVecCAnal = []#np.zeros(iters, float)
EVecAMRAnal = []#np.zeros(iters, float)
EVecFAnal = []#np.zeros(iters, float)
EVecCSolv = []#np.zeros(iters, float)
EVecAMRSolv = []#np.zeros(iters, float)
EVecFSolv = []

print('t_f =', t_f)


t_f = 0.375


In [ ]:
while (2 * t <= t_f):
    t, nt = ST.CalcTime(omegaAMR, CFL, cMatAMR, nt = i)
#     t = (i * t_f) / (t_snaps - 1)
    print('BEFORE: t =', 2 * t)
    # Find initial waveform on coarse, fine, and AMR grids.
    waveInitC = wavesC @ FCoefsC
    waveInitF = wavesF @ FCoefsF
    waveInitAMR = restrictOp @ waveInitF

    # Find Fourier coefficients for initial condition on AMR grid.
    FCoefsAMR = FFTT.FourierCoefs(wavesAMR @ nullspace, waveInitAMR)

    wavePropC = AnalFunc(omegaC, physicsC, WaveFunc, args, 2 * t, deriv = derivative, BooleAve = BooleAve)#, IRT = IRT)#, field = field)#FFTT.FourierCoefs(wavesC, wavePropC) # timePropOpC @ FCoefsC
    wavePropF = AnalFunc(omegaF, physicsF, WaveFunc, args, 2 * t, deriv = derivative, BooleAve = BooleAve)#, IRT = IRT)#, field = field)

    # Find propagated coeficients on coarse and fine grid.
    FCoefsPropC = FFTT.FourierCoefs(wavesC, wavePropC) # FFTT.FourierCoefs(wavesC, wavePropC) # timePropOpC @ FCoefsC
    FCoefsPropF = FFTT.FourierCoefs(wavesF, wavePropF)

    # Find analytically propagated waveform on AMR grid.
    wavePropAMR = AnalFunc(omegaAMR, physicsAMR, WaveFunc, args, 2 * t, deriv = derivative, BooleAve = BooleAve)#, IRT = IRT)#, field = field) # restrictOp @ wavesF @ FCoefsPropF

    # Find analytically propagated coefficients on AMR grid.
    FCoefsPropAMR = FFTT.FourierCoefs(wavesAMR @ nullspace, wavePropAMR)

    # Find numerically propagated coefficients on coarse, finea, and AMR grids.

    solverCoefsC = ST.RungeKutta(omegaC, physicsC, waveInitC, CFL, nt, RK, order, deriv, PropFunc)
    solverCoefsF = ST.RungeKutta(omegaF, physicsF, waveInitF, CFL, 2 * nt, RK, order, deriv, PropFunc)
    solverCoefsAMR = ST.RungeKutta(omegaAMR, physicsAMR, waveInitAMR, CFL, 2 * nt, RK, order, deriv, PropFunc)

    solvPropC = wavesC @ solverCoefsC
    solvPropF = wavesF @ solverCoefsF
    solvPropAMR = wavesAMR @ nullspace @ solverCoefsAMR

    

    # TEST FOR EXACT GAUSSIAN AT NEW LOCATION.

    # Find theoretical propagated coefficients on coarse, fine, and AMR grid. (THIS CAN ONLY BE USED IF MATERIAL IS UNIFORM!)
    # FCoefsPropCTh = FFTT.PropRestrictWaves(omegaC, waveInitC, c * t)
    # FCoefsPropFTh = FFTT.PropRestrictWaves(omegaF, waveInitF, c * t)
    # FCoefsPropAMRTh = FFTT.PropRestrictWaves(omega, waveInitF, c * t)

    # solverCoefsC = ST.RungeKutta(omegaC, physicsC, wavesC, waveInitC, nt, CFL, RK, op = opC) # TimeIntegratorFunc(omegaC, wavesC, waveInitC, nt, cMatC, CFL, DiffFunc)
    # solverCoefsF = ST.RungeKutta(omegaF, physicsF, wavesF, waveInitF, nt, CFL, RK, op = opF) # TimeIntegratorFunc(omegaF, wavesF, waveInitF, nt, cMatF, CFL, DiffFunc)
    # solverCoefsAMR = ST.RungeKutta(omegaAMR, physicsAMR, wavesAMR @ nullspace, waveInitAMR, nt, CFL, RK, op = opAMR) # TimeIntegratorFunc(omegaAMR, wavesAMR @ nullspace, waveInitAMR, nt, cMat, CFL, DiffFunc, order = order)
    # solverCoefsC1 = np.asarray(np.append(solverCoefsC, solverCoefsC))

    # print('')
    # print(np.round(FCoefsPropCTh, 14))
    # print(np.round(FCoefsPropFTh, 14))
    # print(np.round(FCoefsPropAMRTh, 14))


    allCoefsC = PT.Load(FCoefsPropC, solverCoefsC)#, FCoefsPropCTh)
    allCoefsF = PT.Load(FCoefsPropF, solverCoefsF)#, FCoefsPropFTh)
    allCoefsAMR = nullspace @ PT.Load(FCoefsPropAMR, solverCoefsAMR)#, FCoefsPropAMRTh)


    # labels = ['Initial Wave', 'Method of Characteristics Propagated Wave']#, 'RK Propagated Wave']#, 'Rotation Matrix Propagated Wave']
    title = 'filler'
    if ((i % 20 == 0) or (2 * t >= t_f)):
        # print('print t =', t)
        PT.PlotMixedWave(omegaC, physicsC, waves = wavesC, FCoefs = allCoefsC, rescale = [6, 3], xGrid = True, yGrid = True, title = title + 'Coarse' + str(i), dpi = 400, plotCont = plotCont, enlarge = True)
        PT.PlotMixedWave(omegaAMR, physicsAMR, waves = wavesAMR, FCoefs = allCoefsAMR, rescale = [6, 3], xGrid = True, yGrid = True, title = title + 'AMR' + str(i), dpi = 400, plotCont = plotCont, enlarge = True)#, labels = labels, title = 'AMR-Grid Mode Propagation', saveName = 'AMRHansMethod', dpi = 300)
        PT.PlotMixedWave(omegaF, physicsF, waves = wavesF, FCoefs = allCoefsF, rescale = [6, 3], xGrid = True, yGrid = True, title = title + 'Fine' + str(i), dpi = 400, plotCont = plotCont, enlarge = True)#, labels = labels, title = 'Fine-Grid Mode Propagation', saveName = 'Fine', dpi = 300)
    

    if (locs == []):
        wavePropC = np.concatenate((wavePropC, wavePropC / c1))
        wavePropF = np.concatenate((wavePropF, wavePropF / c1))
        wavePropAMR = np.concatenate((wavePropAMR, wavePropAMR / c1))
        solvPropC = np.concatenate((solvPropC, solvPropC / c1))
        solvPropF = np.concatenate((solvPropF, solvPropF / c1))
        solvPropAMR = np.concatenate((solvPropAMR, solvPropAMR / c1))
    
    EVecCAnal = np.append(EVecCAnal, 0.5 * sum(muepsC * wavePropC * wavePropC * hC))
    EVecFAnal = np.append(EVecFAnal, 0.5 * sum(muepsF * wavePropF * wavePropF * hF))
    EVecAMRAnal = np.append(EVecAMRAnal, 0.5 * sum(muepsAMR * wavePropAMR * wavePropAMR * hAMR))
    EVecCSolv = np.append(EVecCSolv, 0.5 * sum(muepsC * solvPropC * solvPropC * hC))
    EVecFSolv = np.append(EVecFSolv, 0.5 * sum(muepsF * solvPropF * solvPropF * hF))
    EVecAMRSolv = np.append(EVecAMRSolv, 0.5 * sum(muepsAMR * solvPropAMR * solvPropAMR * hAMR))
    ntVec = np.append(ntVec, i)
    # print('Energy Coarse:', EVecC)
    # print('Energy AMR:', EVecAMR)
    # print('Energy Fine:', EVecF)
    print('AFTER: t =', 2 * t)
    
    print('')
    i = i + 1
EVecCAnal = np.asarray(EVecCAnal)
EVecFAnal = np.asarray(EVecFAnal)
EVecAMRAnal = np.asarray(EVecAMRAnal)
EVecCSolv = np.asarray(EVecCSolv)
EVecFSolv = np.asarray(EVecFSolv)
EVecAMRSolv = np.asarray(EVecAMRSolv)
ntVec = np.asarray(ntVec)


This is what's happening.
BEFORE: t = 0.0

YOU ARE RUNNING GAUSS! False


YOU ARE RUNNING GAUSS! False


YOU ARE RUNNING GAUSS! False


YOU ARE RUNNING GAUSS! False


YOU ARE RUNNING GAUSS! False


YOU ARE RUNNING GAUSS! False

AFTER: t = 0.0

BEFORE: t = 0.0027621358640099515

YOU ARE RUNNING GAUSS! False


YOU ARE RUNNING GAUSS! False


YOU ARE RUNNING GAUSS! False


YOU ARE RUNNING GAUSS! False


YOU ARE RUNNING GAUSS! False


YOU ARE RUNNING GAUSS! False

You are using WaveEqRHS()!
You are using WaveEqRHS()!
You are using WaveEqRHS()!
You are using WaveEqRHS()!
You are using WaveEqRHS()!
You are using WaveEqRHS()!
You are using WaveEqRHS()!
You are using WaveEqRHS()!
You are using WaveEqRHS()!
You are using WaveEqRHS()!
You are using WaveEqRHS()!
You are using WaveEqRHS()!
You are using WaveEqRHS()!
You are using WaveEqRHS()!
You are using WaveEqRHS()!
You are using WaveEqRHS()!
You are using WaveEqRHS()!
You are using WaveEqRHS()!
You are using WaveEqRHS()!
You are using WaveEqRHS

/Users/sashacurcic/SashasDirectory/ANAG/FV_MG/Modules/PlotTools.py:699: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig, ax = plt.subplots(figsize = size)


AFTER: t = 0.16572815184059708

BEFORE: t = 0.16849028770460703

YOU ARE RUNNING GAUSS! False


YOU ARE RUNNING GAUSS! False


YOU ARE RUNNING GAUSS! False


YOU ARE RUNNING GAUSS! False


YOU ARE RUNNING GAUSS! False


YOU ARE RUNNING GAUSS! False

You are using WaveEqRHS()!
You are using WaveEqRHS()!
You are using WaveEqRHS()!
You are using WaveEqRHS()!
You are using WaveEqRHS()!
You are using WaveEqRHS()!
You are using WaveEqRHS()!
You are using WaveEqRHS()!
You are using WaveEqRHS()!
You are using WaveEqRHS()!
You are using WaveEqRHS()!
You are using WaveEqRHS()!
You are using WaveEqRHS()!
You are using WaveEqRHS()!
You are using WaveEqRHS()!
You are using WaveEqRHS()!
You are using WaveEqRHS()!
You are using WaveEqRHS()!
You are using WaveEqRHS()!
You are using WaveEqRHS()!
You are using WaveEqRHS()!
You are using WaveEqRHS()!
You are using WaveEqRHS()!
You are using WaveEqRHS()!
You are using WaveEqRHS()!
You are using WaveEqRHS()!
You are using WaveEqRHS()!
You are using WaveEqR

In [ ]:
coarse = True

numPoints, font, X, savePath = PT.UsefulPlotVals()

energy = epsilons[0] * sigma * (np.pi**0.5) * np.ones(i, float)

dt = 2 * t / nt
print(t_f, t)
print(i, nt, len(EVecFSolv))


peakCrossing = 3 / (8 * c1 * dt)

EDiffCAnal = abs(EVecCAnal - energy) / energy
EDiffFAnal = abs(EVecFAnal - energy) / energy
EDiffAMRAnal = abs(EVecAMRAnal - energy) / energy

EDiffFAbs = abs(EVecFSolv - energy) / energy
EDiffAMRAbs = abs(EVecAMRSolv - energy) / energy
EDiffCAbs = abs(EVecCSolv - energy) / energy

EDiffFRel = abs(EVecFSolv - EVecFAnal) / energy
EDiffAMRRel = abs(EVecAMRSolv - EVecAMRAnal) / energy
EDiffCRel = abs(EVecCSolv - EVecCAnal) / energy

minY = np.min(EDiffCAnal)
print(minY)
maxY = np.max(EDiffCAnal)

if (minY > np.min(EDiffFAnal)):
    minY = np.min(EDiffFAnal)
if (minY > np.min(EDiffAMRAnal)):
    minY = np.min(EDiffAMRAnal)

if (maxY < np.max(EDiffFAnal)):
    maxY = np.max(EDiffFAnal)
if (maxY < np.max(EDiffAMRAnal)):
    maxY = np.max(EDiffAMRAnal)

if (minY > np.min(EDiffCAbs)):
    minY = np.min(EDiffCAbs)
if (minY > np.min(EDiffFAbs)):
    minY = np.min(EDiffFAbs)
if (minY > np.min(EDiffAMRAbs)):
    minY = np.min(EDiffAMRAbs)

if (maxY < np.max(EDiffCAbs)):
    maxY = np.max(EDiffCAbs)
if (maxY < np.max(EDiffFAbs)):
    maxY = np.max(EDiffFAbs)
if (maxY < np.max(EDiffAMRAbs)):
    maxY = np.max(EDiffAMRAbs)

if (minY > np.min(EDiffCRel)):
    minY = np.min(EDiffCRel)
if (minY > np.min(EDiffFRel)):
    minY = np.min(EDiffFRel)
if (minY > np.min(EDiffAMRRel)):
    minY = np.min(EDiffAMRRel)

if (maxY < np.max(EDiffCRel)):
    maxY = np.max(EDiffCRel)
if (maxY < np.max(EDiffFRel)):
    maxY = np.max(EDiffFRel)
if (maxY < np.max(EDiffAMRRel)):
    maxY = np.max(EDiffAMRRel)



vertX = np.asarray([peakCrossing, peakCrossing])
vertY = np.asarray([0, maxY])
print(peakCrossing, vertX, vertY)

In [ ]:
# if (reflux):
#     reflux = 'reflux'
# else:

# save = True

In [ ]:
saveName = savePath + 'Plot1RK' + str(RK)
fig, ax = plt.subplots(figsize = [12, 10])
plt.title('Analytic Error')
plt.plot(ntVec, EDiffCAnal, color = PT.ColorDefault(0), linestyle = '--', zorder = 0)
plt.plot(ntVec, EDiffAMRAnal, color = PT.ColorDefault(0), linestyle = '-.', zorder = 1)
plt.plot(ntVec, EDiffFAnal, color = PT.ColorDefault(0), linestyle = ':', zorder = 0)
if (locs != []):
    plt.plot(vertX, vertY, color = PT.ColorDefault(0.5), linestyle = '-')
# plt.yscale('log')
ax.grid(True, zorder = 0)
ax.set_axisbelow(True)
fig.canvas.draw()
if (save):
    fig.savefig(saveName + '.png', bbox_inches = 'tight', dpi = 400, transparent = False)
plt.show()

In [ ]:
saveName = savePath + 'Plot2RK' + str(RK)
fig, ax = plt.subplots(figsize = [12, 10])
plt.title('Absolute (blue) versus Relative (orange) Error (LMR and Fine)')
plt.plot(ntVec, EDiffAMRAbs, color = PT.ColorDefault(0), linestyle = '-.', zorder = 1)
plt.plot(ntVec, EDiffAMRRel, color = PT.ColorDefault(1), linestyle = '-.', zorder = 1)
plt.plot(ntVec, EDiffFAbs, color = PT.ColorDefault(0), linestyle = ':', zorder = 0)
plt.plot(ntVec, EDiffFRel, color = PT.ColorDefault(1), linestyle = ':', zorder = 0)
if (locs != []):
    plt.plot(vertX, vertY, color = PT.ColorDefault(0.5), linestyle = '-')
# plt.yscale('log')
ax.grid(True, zorder = 0)
ax.set_axisbelow(True)
fig.canvas.draw()
if (save):
    fig.savefig(saveName + '.png', bbox_inches = 'tight', dpi = 400, transparent = False)
plt.show()

In [ ]:
saveName = savePath + 'Plot3RK' + str(RK)
fig, ax = plt.subplots(figsize = [12, 10])
plt.title('Absolute (blue) versus Relative (orange) Error')
plt.plot(ntVec, EDiffAMRAbs, color = PT.ColorDefault(0), linestyle = '-.', zorder = 1)
plt.plot(ntVec, EDiffAMRRel, color = PT.ColorDefault(1), linestyle = '-.', zorder = 1)
plt.plot(ntVec, EDiffFAbs, color = PT.ColorDefault(0), linestyle = ':', zorder = 0)
plt.plot(ntVec, EDiffFRel, color = PT.ColorDefault(1), linestyle = ':', zorder = 0)
# plt.yscale('log')
ax.grid(True, zorder = 0)
ax.set_axisbelow(True)
fig.canvas.draw()
if (save):
    fig.savefig(saveName + '.png', bbox_inches = 'tight', dpi = 400, transparent = False)
plt.show()

In [ ]:
saveName = savePath + 'Plot4RK' + str(RK)
fig, ax = plt.subplots(figsize = [12, 10])
plt.title('Absolute Error (LMR and Fine)')
plt.plot(ntVec, EDiffAMRRel, color = PT.ColorDefault(1), linestyle = '-.', zorder = 1)
plt.plot(ntVec, EDiffFRel, color = PT.ColorDefault(1), linestyle = ':', zorder = 0)
# plt.yscale('log')
ax.grid(True, zorder = 0)
ax.set_axisbelow(True)
fig.canvas.draw()
if (save):
    fig.savefig(saveName + '.png', bbox_inches = 'tight', dpi = 400, transparent = False)
plt.show()

In [ ]:
saveName = savePath + 'Plot5RK' + str(RK)
fig, ax = plt.subplots(figsize = [12, 10])
plt.title('Absolute Error (Fine)')
plt.plot(ntVec, EDiffFRel, color = PT.ColorDefault(1), linestyle = ':', zorder = 0)
# plt.yscale('log')
ax.grid(True, zorder = 0)
ax.set_axisbelow(True)
fig.canvas.draw()
if (save):
    fig.savefig(saveName + '.png', bbox_inches = 'tight', dpi = 400, transparent = False)
plt.show()

In [ ]:
saveName = savePath + 'Plot6RK' + str(RK)
fig, ax = plt.subplots(figsize = [12, 10])
plt.title('Absolute (blue) versus Relative (orange) Error (Coarse, LMR, and Fine)')
plt.plot(ntVec, EDiffCAbs, color = PT.ColorDefault(0), linestyle = '--', zorder = 0)
plt.plot(ntVec, EDiffCRel, color = PT.ColorDefault(1), linestyle = '--', zorder = 0)
plt.plot(ntVec, EDiffAMRAbs, color = PT.ColorDefault(0), linestyle = '-.', zorder = 1)
plt.plot(ntVec, EDiffAMRRel, color = PT.ColorDefault(1), linestyle = '-.', zorder = 1)
plt.plot(ntVec, EDiffFAbs, color = PT.ColorDefault(0), linestyle = ':', zorder = 0)
plt.plot(ntVec, EDiffFRel, color = PT.ColorDefault(1), linestyle = ':', zorder = 0)
if (locs != []):
    plt.plot(vertX, vertY, color = PT.ColorDefault(0.5), linestyle = '-')
# plt.yscale('log')
ax.grid(True, zorder = 0)
ax.set_axisbelow(True)
fig.canvas.draw()
if (save):
    fig.savefig(saveName + '.png', bbox_inches = 'tight', dpi = 400, transparent = False)
plt.show()

In [ ]:
saveName = savePath + 'Plot7RK' + str(RK)
fig, ax = plt.subplots(figsize = [12, 10])
plt.title('Absolute Error (Coarse, LMR, and Fine)')
plt.plot(ntVec, EDiffAMRRel, color = PT.ColorDefault(1), linestyle = '-.', zorder = 1)
plt.plot(ntVec, EDiffCRel, color = PT.ColorDefault(1), linestyle = '--', zorder = 0)
plt.plot(ntVec, EDiffFRel, color = PT.ColorDefault(1), linestyle = ':', zorder = 0)
# plt.yscale('log')
ax.grid(True, zorder = 0)
ax.set_axisbelow(True)
fig.canvas.draw()
if (save):
    fig.savefig(saveName + '.png', bbox_inches = 'tight', dpi = 400, transparent = False)
plt.show()

In [ ]:
saveName = savePath + 'Plot8RK' + str(RK)
fig, ax = plt.subplots(figsize = [12, 10])
plt.title('Absolute Error (Coarse and Fine)')
plt.plot(ntVec, EDiffCRel, color = PT.ColorDefault(1), linestyle = '--', zorder = 0)
plt.plot(ntVec, EDiffFRel, color = PT.ColorDefault(1), linestyle = ':', zorder = 0)
# plt.yscale('log')
ax.grid(True, zorder = 0)
ax.set_axisbelow(True)
fig.canvas.draw()
if (save):
    fig.savefig(saveName + '.png', bbox_inches = 'tight', dpi = 400, transparent = False)
plt.show()

In [ ]:
print(EDiffCRel)
print(EDiffFRel)
print(EDiffAMRRel)

In [ ]:
print('Coarse Max Norm:', np.max(EDiffCRel))
print('')
print('LMR Max Norm:', np.max(EDiffAMRRel))
print('')
print('Fine Max Norm:', np.max(EDiffFRel))
print('')

In [ ]:
X = []

for q in range(11):
    X = np.append(X, q)
half = (10 // 2) + 1
print(X)
print(X[::2])
print(X[:half])

In [ ]:
print(c1, c2)